# Урок 20. Условные функции и статистика в таблицах

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-21.ipynb)

---

СУММЕСЛИ, СЧЁТЕСЛИ, СРЗНАЧЕСЛИ, вложенные ЕСЛИ. Расчёт долей и процентов. Практика в Excel, проверка расчёта вторым способом.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-20", name=ФИО, klass=Класс)

## Разбираемся

### Считать не всё, а нужное

`СУММ` и `СРЗНАЧ` считают весь диапазон целиком. Но настоящие вопросы
звучат иначе: сколько **пятёрок** в 9А, какая средняя цена
**у книг дороже 500**, сколько заказов **из Москвы**.

Это условные функции, и их всего три плюс одна.

### Три функции с условием

| Функция | Что считает |
|---|---|
| `=СЧЁТЕСЛИ(диапазон; условие)` | сколько ячеек подходит |
| `=СУММЕСЛИ(диапазон; условие; суммируемый)` | сумму подходящих |
| `=СРЗНАЧЕСЛИ(диапазон; условие; усредняемый)` | среднее подходящих |

Условие пишут либо значением, либо **строкой со знаком**:

```
=СЧЁТЕСЛИ(B2:B100; 5)          пятёрок
=СЧЁТЕСЛИ(B2:B100; ">3")       больше трёх
=СЧЁТЕСЛИ(A2:A100; "9А")       из девятого А
=СЧЁТЕСЛИ(A2:A100; "Ив*")      фамилии на «Ив»
```

Знаки сравнения и звёздочку берут **в кавычки** — иначе таблица
не поймёт, где кончается условие. Это ловушка номер один.

### Третий аргумент

У `СУММЕСЛИ` три аргумента, и их путают чаще всего:

```
=СУММЕСЛИ(A2:A100; "9А"; C2:C100)
           ↑          ↑     ↑
     где искать   что искать  что сложить
```

«Пройди по столбцу A, найди строки с 9А, сложи из них столбец C».
Если третий аргумент не указать, сложится сам столбец A.

### Условие по нескольким столбцам

Когда условий несколько, к имени добавляется `МН` — «множественное»:

```
=СЧЁТЕСЛИМН(A2:A100; "9А"; B2:B100; 5)
```

Читается: «строк, где класс 9А **и** оценка 5». Условия соединяются
только через «и»; для «или» складывают два СЧЁТЕСЛИМН.

### Вложенные ЕСЛИ

Когда исходов больше двух, `ЕСЛИ` вкладывают друг в друга:

```
=ЕСЛИ(A1>=85; "5"; ЕСЛИ(A1>=70; "4"; ЕСЛИ(A1>=50; "3"; "2")))
```

Это ровно `if / elif / elif / else` из Python, записанный в строку.
И у него та же ловушка: **порядок условий**. Начнёшь с `A1>=50` —
и девяносто баллов получат тройку, потому что первое условие
подошло раньше.

Правило то же, что в программировании: от самого строгого условия
к самому мягкому.

### Доли и проценты

Доля — это часть, делённая на целое:

```
=C2/$C$10          доля строки в итоге
```

Знаменатель закрепляют долларами, иначе при копировании формула
начнёт делить на что попало.

Чтобы получить проценты, не нужно умножать на 100: достаточно
применить к ячейке процентный формат, и `0,25` покажется как `25 %`.
Умножение на 100 **и** процентный формат дадут 2500 % — ловушка
номер два.

### Зачем проверять расчёт кодом

Формулу в таблице легко растянуть не на тот диапазон, и ошибка
ничем себя не выдаёт: число есть, выглядит правдоподобно.

Поэтому серьёзный расчёт проверяют **вторым способом**: та же
статистика, посчитанная программой, должна совпасть с таблицей.
Не совпало — ищем, кто из двоих врёт. Этим и займёмся.

## Практическая работа. Статистика по баллам

[Скачайте файл с баллами](https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/files/excel/g09-20-bally.xlsx) и откройте
в **Excel**. В нём 60 учеников трёх девятых классов и баллы
за пробный экзамен.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-dannye.png" width="900" alt="Файл с баллами">

*Файл с баллами*

Все итоги будем писать справа: подпись в столбце F, формула —
в G. Сделайте столбец F пошире.

### Шаг 1. СЧЁТЕСЛИ — сколько подходит

В F2 — `Учеников 9А`. В G2 начните `=СЧЁТЕСЛИ(` и нажмите **fx**
слева от строки формул. Откроется окно **Аргументы функции**:

* **Диапазон** — `A2:A61`, где искать;
* **Критерий** — `9А`, что искать. Кавычки Excel поставит сам.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-schetesli.png" width="900" alt="СЧЁТЕСЛИ в окне аргументов">

*СЧЁТЕСЛИ в окне аргументов*

**ОК**. В F6 — `Выше 70 баллов`, в G6 напишите сразу в ячейке:
`=СЧЁТЕСЛИ(C2:C61;">70")`. Знак сравнения — **в кавычках**,
иначе формула не примется.

### Шаг 2. СУММЕСЛИ — три аргумента

В F3 — `Сумма баллов 9Б`. В G3 — `=СУММЕСЛИ(` и **fx**:

* **Диапазон** — `A2:A61`, где искать класс;
* **Критерий** — `9Б`;
* **Диапазон_суммирования** — `C2:C61`, что складывать.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-summesli.png" width="900" alt="СУММЕСЛИ в окне аргументов">

*СУММЕСЛИ в окне аргументов*

Третий аргумент забывают чаще всего. Без него Excel попробует
сложить сам столбец классов — и получит ноль.

В F4 — `Средний балл 9В`, в G4: `=СРЗНАЧЕСЛИ(A2:A61;"9В";C2:C61)`.
Аргументы те же, что у СУММЕСЛИ. Оставьте один знак после запятой.

### Шаг 3. СЧЁТЕСЛИМН — несколько условий

В F5 — `В 9Б от 85 баллов`. В G5 — `=СЧЁТЕСЛИМН(` и **fx**.
Аргументы идут парами «диапазон — условие»: новая пара
появляется, как только заполните предыдущую.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-scheteslimn.png" width="900" alt="СЧЁТЕСЛИМН в окне аргументов">

*СЧЁТЕСЛИМН в окне аргументов*

Все условия соединяются через **и**. Для «или» складывают два
СЧЁТЕСЛИ.

Проверьте блок итогов в режиме **Формулы → Показать формулы**:

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-itogi-formuly.png" width="900" alt="Итоги в виде формул">

*Итоги в виде формул*

### Шаг 4. Вложенные ЕСЛИ — балл в оценку

В D2 напишите:

```
=ЕСЛИ(C2>=85;5;ЕСЛИ(C2>=70;4;ЕСЛИ(C2>=50;3;2)))
```

и растяните до D61 (двойной щелчок по маркеру заполнения).
Скобок в конце три — по числу ЕСЛИ.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-esli.png" width="900" alt="Вложенные ЕСЛИ">

*Вложенные ЕСЛИ*

А теперь нарочно переставьте условия: `=ЕСЛИ(C2>=50;3;ЕСЛИ(C2>=70;4;
ЕСЛИ(C2>=85;5;2)))`. У 92 баллов — тройка: первое же условие
подошло, до остальных дело не дошло.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-esli-oshibka.png" width="900" alt="Неправильный порядок условий">

*Неправильный порядок условий*

Верните правильную формулу. В F7 — `Пятёрок`, в G7 —
`=СЧЁТЕСЛИ(D2:D61;5)`.

### Шаг 5. Доли и проценты

Добавьте лист `Доли`. Впишите:

| | A | B | C |
|---|---|---|---|
| 1 | Класс | Сумма баллов | Доля |
| 2 | 9А | `=СУММЕСЛИ(Баллы!A:A;A2;Баллы!C:C)` | `=B2/$B$5` |
| 3 | 9Б | растянуть | растянуть |
| 4 | 9В | растянуть | растянуть |
| 5 | Всего | `=СУММ(B2:B4)` | `=СУММ(C2:C4)` |

`Баллы!A:A` — весь столбец A **с другого листа**: имя листа,
восклицательный знак, адрес. Проще всего не печатать, а во время
ввода формулы щёлкнуть по ярлычку «Баллы» и выделить столбец мышкой.

Критерий здесь — **ячейка** A2, а не текст: растянули вниз —
и формула сама берёт 9Б, 9В.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-dolya.png" width="900" alt="Доли — дробными числами">

*Доли — дробными числами*

Выделите C2:C5 и нажмите **%** (**Процентный формат**) на вкладке
**Главная**. Умножать на 100 не нужно: `0,328` и есть `32,8 %`.
Если умножить **и** поставить формат, получится 3280 %.

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/excel/g09-20-procenty.png" width="900" alt="Проценты">

*Проценты*

Внизу в столбце «Доля» должно получиться ровно **100 %**. Не 100 —
значит, диапазоны в формулах разъехались. Это и есть проверка
вторым способом: разные пути к одному числу обязаны совпасть.

## Проверяем себя

Все ответы — из вашего файла Excel.

### Задача 1. СЧЁТЕСЛИ

Сколько учеников в 9А?

In [ ]:
#@title 🧩 Задача 1. Учеников 9А { display-mode: "form" }
#@markdown Впиши число из G2
в_9а = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", в_9а, "f5ca38f748a1d6ea",
         hint='=СЧЁТЕСЛИ(A2:A61;"9А")')

### Задача 2. СУММЕСЛИ

Чему равна сумма баллов 9Б?

In [ ]:
#@title 🧩 Задача 2. Сумма 9Б { display-mode: "form" }
#@markdown Впиши число из G3
сумма_9б = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", сумма_9б, "e3e69eaf6c365e40",
         hint="Если получился 0 — забыт третий аргумент.")

### Задача 3. СРЗНАЧЕСЛИ

Какой средний балл в 9В? Один знак после запятой, через запятую.

In [ ]:
#@title 🧩 Задача 3. Средний 9В { display-mode: "form" }
#@markdown Впиши число из G4
среднее_9в = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("3", среднее_9в, "3e7959225c9f08ec",
         hint="Кнопка «Уменьшить разрядность» оставит один знак.")

### Задача 4. СЧЁТЕСЛИМН

Сколько учеников 9Б набрали 85 баллов и больше?

In [ ]:
#@title 🧩 Задача 4. Сильные в 9Б { display-mode: "form" }
#@markdown Впиши число из G5
сильных = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("4", сильных, "e7f6c011776e8db7",
         hint='Условие «от 85» пишется как ">=85".')

### Задача 5. Вложенные ЕСЛИ

Сколько пятёрок получилось во всех трёх классах?

In [ ]:
#@title 🧩 Задача 5. Пятёрки { display-mode: "form" }
#@markdown Впиши число из G7
пятёрок = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("5", пятёрок, "4fc82b26aecb47d2",
         hint="Проверьте порядок условий: от 85 — первым.")

### Задача 6. Доля в процентах

Какую долю от всех баллов набрал 9А? В процентах, один знак после
запятой, без знака %.

In [ ]:
#@title 🧩 Задача 6. Доля 9А { display-mode: "form" }
#@markdown Впиши число из C2 листа «Доли»
доля_9а = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("6", доля_9а, "2a4628c61253abf6",
         hint="Процентный формат и одна цифра после запятой.")

### Задача 7. Условие со знаком

Сколько учеников набрали больше 70 баллов?

In [ ]:
#@title 🧩 Задача 7. Больше 70 { display-mode: "form" }
#@markdown Впиши число из G6
выше_70 = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("7", выше_70, "5f9c4ab08cac7457",
         hint='">70" — строго больше, в кавычках.')

## Домашнее задание

### Домашнее задание 1. Статистика в таблице и в коде

В файле с баллами добавьте лист `ДЗ` и посчитайте формулами
(каждый итог — с подписью слева):

1. средний балл по всем 60 ученикам;
2. сколько учеников набрали **выше среднего** — СЧЁТЕСЛИ,
   где условие собрано из знака и ячейки: `">"&B1`;
3. сколько двоек в каждом классе — три СЧЁТЕСЛИМН (класс и оценка
   из столбца D);
4. доля двоек каждого класса **от числа учеников этого класса**,
   в процентах;
5. в каком классе доля двоек больше — ответ словами в ячейке.

Затем проверьте кодом пункт 3 для одного класса. Скопируйте из
Excel баллы этого класса, вставьте в ячейку кода в список
`баллы = [...]` (переводы строк замените запятыми) и циклом
со счётчиком посчитайте, сколько баллов меньше 50. Число должно
совпасть с таблицей; не совпало — ищите, кто ошибся.

Сдайте учителю файл Excel и скриншот ячейки с кодом.

### Домашнее задание 2. Сколько подходит под два условия

Функция возвращает количество строк, где класс совпадает
**и** балл строго меньше порога. Журнал — список строк
`[класс, фамилия, балл]`. Это тот же СЧЁТЕСЛИМН, только циклом.

In [ ]:
def слабых(журнал, класс, порог):
    return ...

In [ ]:
si.check("дз2", слабых, [
    (([["9А", "И", 40], ["9А", "П", 80], ["9Б", "С", 30]], "9А", 50), 1),
    (([["9А", "И", 40], ["9А", "П", 30]], "9А", 50), 2),
    (([["9А", "И", 90]], "9А", 50), 0),
])

### Домашнее задание 3. Сколько это процентов

Сколько процентов составляет 7 от 28?

In [ ]:
#@title 🏠 Домашнее 3. Проценты { display-mode: "form" }
#@markdown Впиши число процентов
процентов = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", процентов, "b7a56873cd771f2c",
         hint="Раздели и умножь на сто.")

---

### Любопытно

Проценты умеют путать даже взрослых. Заголовок «доля отказов выросла
с 2 % до 4 %» можно честно написать двумя способами: «выросла
на 2 процентных пункта» и «выросла на 100 %». Оба верны, но второй
звучит как катастрофа.

Поэтому в статистике различают **процент** и **процентный пункт**,
а в новостях — почти никогда. Теперь вы умеете проверять такие
заголовки: делите одно на другое и смотрите, о чём речь.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 19](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-19.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 21 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-21.ipynb)